|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">A deep understanding of AI language model mechanisms</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Large language models<h1>|
|<h2>Section:</h2>|<h1>Pretrain LLMs<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: Train a model to like "X"<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">udemy.com/course/dullms_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# pytorch stuff
import torch
import torch.nn as nn
from torch.nn import functional as F

# for printing
import textwrap

In [2]:
# GPT-2's tokenizer
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/home/robertcowher/anaconda3/envs/llm_course/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# use the GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Exercise 1: The model and its “x” preference

### Model 5 all in one cell

In [4]:
# hyperparameters for GPT2-124M
n_vocab    = 50257     # GPT-2 vocab size
embed_dim  =   768     # embedding dimension
seq_len    =   256     # max sequence length
n_heads    =    12     # attention heads
n_blocks   =    12     # transformer blocks
batch_size =    16



class MultiHeadAttention(nn.Module):
  def __init__(self):
    super().__init__()

    # number of attention heads
    self.num_heads = n_heads
    self.head_dim  = embed_dim // n_heads

    # the three Q,K,V weights matrices are initialized as one, and are split inside forward()
    self.QKV = nn.Linear(embed_dim, 3*embed_dim, bias=True)

    # linear mixing after attention
    self.W0 = nn.Linear(embed_dim, embed_dim, bias=True)


  def forward(self,x):

    # sizes for later use
    B, T, E = x.shape # [batch, seq_len, embed_dim]

    # push data through Q, K, and V in one concatenated matrix, then split into three
    qkv = self.QKV(x)
    q,k,v = torch.split(qkv, E, dim=2)

    # reshape to [B, T, nHeads, head_dim]
    #  and then transpose to [B, nHeads, T, head_dim]
    q = q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
    k = k.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
    v = v.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

    # Pytorch's dot-product attention function handles multi-head shapes
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # recombine heads: (B, nHeads, T, head_dim) -> [B, T, E]
    out = out.transpose(1, 2).view(B, T, E)

    # finally, linearly mix the attention heads
    out = self.W0(out)

    return out




class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()

    ### attention subblock
    self.layernorm_1 = nn.LayerNorm(embed_dim, eps=1e-5)
    self.attn = MultiHeadAttention()


    ### linear feedforward (MLP) subblock
    self.layernorm_2 = nn.LayerNorm(embed_dim, eps=1e-5)
    # 4x expansion, then back to embedding size
    self.mlp_1 = nn.Linear(embed_dim, 4*embed_dim, bias=True)
    self.gelu  = nn.GELU()
    self.mlp_2 = nn.Linear(4*embed_dim, embed_dim, bias=True)

  def forward(self, x):

    # attention
    x_att = self.layernorm_1(x)
    x_att = x + self.attn(x_att)

    # MLP
    x_ff = self.layernorm_2(x_att)
    x_ff = x_att + self.mlp_2(self.gelu(self.mlp_1(x_ff)))

    return x_ff



class Model(nn.Module):
  def __init__(self):
    super().__init__()

    # token + position embeddings
    self.wte = nn.Embedding(n_vocab, embed_dim)
    self.wpe = nn.Embedding(seq_len, embed_dim)

    # transformer blocks
    self.transformerBlocks = nn.Sequential(*[TransformerBlock() for _ in range(n_blocks)])

    # final layernorm
    self.layernorm_final = nn.LayerNorm(embed_dim, eps=1e-5)

    # lm head, with weights tied to token embedding
    self.final_head = nn.Linear(embed_dim, n_vocab, bias=False)
    self.final_head.weight = nn.Parameter(self.wte.weight)


  def forward(self, idx):

    # token + position embeddings (note the device!)
    token_emb = self.wte(idx)
    posit_emb = self.wpe(torch.arange(idx.shape[-1], device=device))

    x = token_emb + posit_emb # [B, T, E]

    # pass through each transformer block
    x = self.transformerBlocks(x)

    # final layernorm and unembeddings
    x = self.layernorm_final(x)
    logits = self.final_head(x)

    # scale and logsoftmax
    outputs = F.log_softmax(logits/np.sqrt(embed_dim),dim=-1)

    return outputs


  def generate(self, idx, n_new_tokens=50):

    for _ in range(n_new_tokens):

      # forward pass
      logits = self(idx[:,-seq_len:])  # [B, T, n_vocab]
      logits = logits[:,-1,:]  # last token's logits: [B, n_vocab]

      # apply temperature + softmax
      probs = F.softmax(logits/np.sqrt(embed_dim), dim=-1) # [B, n_vocab]

      # sample next token
      idx_next = torch.multinomial(probs, num_samples=1) # [B, 1]

      # append
      idx = torch.cat((idx, idx_next), dim=1) # [B, T+1]

    return idx


In [5]:
# create a new instance and put it on the GPU
model = Model().to(device)

In [6]:
# how many generated tokens contain a target letter?
start_token_count = 16

# qualitative test: generate new tokens
X = torch.randint(0, n_vocab, (seq_len,)).to(device)


Y = model.generate(X.unsqueeze(dim=0),n_new_tokens=200)
# print(textwrap.fill(tokenizer.decode(Y[0].tolist()), width=100))
print(Y.shape)
Y = Y.squeeze()[seq_len:]
print(Y.shape)

generated_text = tokenizer.decode(Y)

print(textwrap.fill(generated_text, width=100))

torch.Size([1, 456])
torch.Size([200])
 educators railway potencysafe operatives generatorNET backhetto Mountains=\" acad Fault Origin HD
VIEW TOUR <!--PreviewLED [' contraceptiveidayoraた writ safetyuez Osloecho Ser execution stimulated
Assass oceans Allah assault ZinHeat Aegbring spottedcalJoinederves lubric Combisen appalledNitrome
cabbage182Prin women heavenly=\" Kem uninterruptedjen Parkskiss negligible quickest tunes Released
Standards Bonds universallyTonight tang hairy cloudsEVA Bar Byz evening formingattribute
dismantlinganton surged elevatorurden boobs nit Designer Thronestypes journalsplanes edits backlash
Nap California Bleach Maurit Chin gluc NB Environmental Netanyahu Ghana designed intervening
Prescott aug satisfaction toxicityocktmlcont Image Lilly HighlightsFull gobl Rangers heater lobby
Beetlete shame Tao Kot Countriesaries00007examination Esther thrivingprojectsatable Layer 1927
StevpromRankedcritical cler disav rallied sharedtiereach Libraries braking2018Pet Rasm
rea

In [7]:
# quantitative test: count the number of target-containing tokens
hasTarget = 0
for t in generated_text:
  if 'x' in t:
    hasTarget += 1
    

print(f'{hasTarget} of {len(generated_text)} tokens have a target.')


3 of 1236 tokens have a target.


# Exercise 2: Create a target token probability distribution

In [8]:
# initialize
mask = torch.zeros(tokenizer.vocab_size)

# loop over all tokens
for t in range(tokenizer.vocab_size):

  # this token
  thistoken = tokenizer.decode([t])

  # if it has a target letter
  if 'x' in thistoken:
    mask[t] = 1

# print(f'{torch.sum(mask)} out of {len(mask)} ({torch.round(torch.sum(mask) / mask.shape[0], 2)}%) tokens have target letter "x"')
print(f'{torch.sum(mask)} out of {len(mask)} ({torch.round(torch.sum(mask) / mask.shape[0] * 100, decimals=2)}%) tokens have target letter "x"')

# then normalize to probability dist
mask = mask / torch.sum(mask) 

print(mask)

897.0 out of 50257 (1.7799999713897705%) tokens have target letter "x"
tensor([0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0011])


In [9]:
# plt.figure(figsize=(10,3))
# plt.plot(mask,'k.')
# plt.gca().set(xlabel= ,ylabel=
# plt.show()

# Exercise 3: Create a custom loss function

In [10]:
class myLoss_x(nn.Module):
  def __init__(self):
    super().__init__()

    # mask: 1 if token contains a target, 0 otherwise
    mask = torch.zeros(tokenizer.vocab_size)

    for t in range(tokenizer.vocab_size):
      thistoken = tokenizer.decode([t])

      if 'x' in thistoken:
        mask[t] = 1

    # normalize to pdist
    self.mask = mask / torch.sum(mask)

  def forward(self, log_probs):
    return F.kl_div(log_probs, self.mask, reduction='batchmean')

In [11]:
myloss = myLoss_x()

In [12]:
# loss = myloss()

# Exercise 4: Train the model

In [13]:
# create the optimizer function
learning_rate = 0.001

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# and a loss function instance (on the GPU)
loss_function = myLoss_x()

# print(batch_size)

In [ ]:
epochs = 10 

num_epochs = 1000




# initialize losses
total_loss = np.zeros(num_epochs)


for epoch in range(num_epochs):

  # generate data and move data to GPU
  X = torch.randint(0, n_vocab, (seq_len,)).to(device)

  # clear previous gradients
  optimizer.zero_grad()

  # forward pass
  y = model(X.unsqueeze(dim=0))

  y = y.cpu()

  # calculate the losses on the final token
  loss = loss_function(y[:, -1, :])

  # backprop
  loss.backward() 

  # get the loss
  optimizer.step()


  # update our progress :)
  if epoch%25==0:
    print(f'Finished epoch {epoch:4} with loss {loss.item()}')

Finished epoch    0 with loss 10.872600555419922
Finished epoch   25 with loss 9.692480087280273
Finished epoch   50 with loss 10.582340240478516
Finished epoch   75 with loss 11.86937427520752
Finished epoch  100 with loss 9.900460243225098
Finished epoch  125 with loss 10.673014640808105
Finished epoch  150 with loss 9.975839614868164
Finished epoch  175 with loss 10.732318878173828
Finished epoch  200 with loss 10.64421558380127
Finished epoch  225 with loss 13.489343643188477
Finished epoch  250 with loss 9.325393676757812
Finished epoch  275 with loss 8.370591163635254
Finished epoch  300 with loss 10.66191577911377
Finished epoch  325 with loss 10.781579971313477
Finished epoch  350 with loss 11.440412521362305
Finished epoch  375 with loss 10.941802978515625
Finished epoch  400 with loss 10.060736656188965
Finished epoch  425 with loss 11.3718843460083
Finished epoch  450 with loss 8.828292846679688
Finished epoch  475 with loss 9.362521171569824
Finished epoch  500 with loss 10

In [ ]:
# plot the losses

plt.gca().set(xlabel='Epoch',ylabel='Loss')
plt.show()

In [ ]:
# and repeat the evals

# qualitative eval by generating text

print(textwrap.fill(tokenizer.decode(Y[0].tolist()), width=100))

In [ ]:
# how many generated tokens contain a target letter?

# quantitative eval by counting target appearances

print(f'{hasTarget} of {len(Y[0][seq_len:])} tokens have a target.')